# Probability Review for Machine Learning

**Reference:** CS229 Probability Review — Kolter & Do, Stanford

---

## Why Probability?

The world is uncertain. A photo might be a cat or a dog — you're not 100% sure. A medical test might be positive — but that doesn't mean you have the disease. A self-driving car sees an object ahead — is it a pedestrian or a shadow?

Machine learning is the science of making good decisions under uncertainty. Probability is the language it uses.

Every major concept in ML — loss functions, training, Bayes classifiers, VAEs, RLHF — is built on probability. This notebook builds that foundation from scratch.

**8 concepts. Each one: intuition first → formula → code → ML connection.**

| # | Concept | One Line |
|---|---|---|
| 1 | Random Variables | A number whose value we don't know yet |
| 2 | Probability Distributions | How probability is spread across possible values |
| 3 | Bayes' Theorem | How to update your belief when you see new evidence |
| 4 | Expectation and Variance | The average and the spread |
| 5 | Joint, Marginal, Conditional | Probability involving two things at once |
| 6 | Maximum Likelihood Estimation | Find the parameters that best explain your data |
| 7 | KL Divergence | Measure how different two distributions are |
| 8 | Central Limit Theorem | Why everything tends to look Gaussian |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Libraries loaded.")

---
## 1. Random Variables

### Intuition

Before you roll a die, you don't know what number will come up. That unknown number is a **random variable** — a variable whose value is determined by a random process.

We use a capital letter like $X$ to name the random variable, and a lowercase letter like $x$ to name a specific value it takes.

**Example:**  
Roll a die. $X$ = the result. $X$ could be 1, 2, 3, 4, 5, or 6. Each with probability 1/6.  
We write: $P(X = 3) = 1/6$

### Two Types

**Discrete** — countable outcomes. You can list them.  
Examples: die roll {1,2,3,4,5,6} · coin flip {heads, tails} · class label {cat, dog, bird}

**Continuous** — any value in a range. You can't list them all.  
Examples: temperature · height · pixel intensity · a model's confidence score

### The Two Rules

For any random variable, two rules always hold:
1. Every probability is between 0 and 1: $0 \leq P(X=x) \leq 1$
2. All probabilities sum to 1: $\sum_x P(X=x) = 1$

Rule 2 means: *something* must happen.

### Formula

**Discrete — Probability Mass Function (PMF):**
$$P(X = x) \quad \text{— the probability that X takes value x}$$

**Continuous — Probability Density Function (PDF):**
$$p(x) \quad \text{such that} \quad \int_{-\infty}^{\infty} p(x)\,dx = 1$$

For continuous variables, $p(x)$ is not a probability — it's a *density*. The probability of falling in a range $[a,b]$ is $\int_a^b p(x)dx$.

### In ML

A classifier's output is a random variable. For a 10-class image classifier:
$$P(Y = \text{cat} \mid \text{image}) = 0.87$$

The model doesn't say "this is a cat" — it says "I believe this is a cat with 87% probability." That's a random variable.

In [ ]:
outcomes = np.arange(1, 7)
probabilities = np.ones(6) / 6

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(outcomes, probabilities, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Outcome x')
axes[0].set_ylabel('P(X = x)')
axes[0].set_title('Discrete RV: Fair Die\nProbability Mass Function')
axes[0].set_ylim(0, 0.25)

x = np.linspace(140, 210, 300)
pdf = stats.norm.pdf(x, loc=170, scale=10)
axes[1].plot(x, pdf, color='coral', linewidth=2)
axes[1].fill_between(x, pdf, alpha=0.3, color='coral')
axes[1].set_xlabel('Height (cm)')
axes[1].set_ylabel('Density p(x)')
axes[1].set_title('Continuous RV: Height\nProbability Density Function')

plt.tight_layout()
plt.show()

print(f"Sum of all probabilities (die): {probabilities.sum():.1f} — must equal 1")
print(f"Integral of PDF (height): {np.trapz(pdf, x):.4f} — must equal 1")

---
## 2. Probability Distributions

### Intuition

A **distribution** is the full picture of how probability is spread across all possible values. Not just one probability — the entire shape.

In ML, the choice of distribution is not arbitrary — it encodes your assumption about what kind of values your data can take. Your choice of distribution determines your loss function.

### Three Distributions You Must Know

**Gaussian (Normal)** — the bell curve. For continuous values that cluster around a mean.

Two parameters: $\mu$ (mean — where the center is) and $\sigma$ (standard deviation — how wide it is).
$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

The exponent tells you: the further $x$ is from $\mu$, the smaller the probability.

---

**Bernoulli** — exactly two outcomes: 0 or 1. One parameter: $p$ = probability of outcome 1.
$$P(X=1) = p \qquad P(X=0) = 1-p$$
Example: spam or not spam. Positive or negative test result.

---

**Categorical** — multiple outcomes (more than 2). One probability per class.
$$P(X=k) = p_k \qquad \sum_k p_k = 1$$
Example: which of 10 CIFAR-10 classes is this image?

### In ML

| Distribution | Used for | Loss function |
|---|---|---|
| Gaussian | Regression, noise | Mean Squared Error |
| Bernoulli | Binary classification | Binary Cross-Entropy |
| Categorical | Multi-class classification | Cross-Entropy + Softmax |

**Your choice of loss function is your choice of distribution. They are the same thing.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gaussian — different mu and sigma
x = np.linspace(-6, 6, 300)
for mu, sigma, label in [(0, 1, 'μ=0, σ=1'), (0, 2, 'μ=0, σ=2'), (2, 1, 'μ=2, σ=1')]:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), label=label, linewidth=2)
axes[0].set_title('Gaussian Distribution')
axes[0].set_xlabel('x')
axes[0].set_ylabel('p(x)')
axes[0].legend()

# Bernoulli
for p, offset in [(0.3, -0.15), (0.7, 0), (0.9, 0.15)]:
    axes[1].bar([0 + offset, 1 + offset], [1-p, p], width=0.12, label=f'p={p}', alpha=0.8)
axes[1].set_title('Bernoulli Distribution')
axes[1].set_xlabel('Outcome')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['0 (negative)', '1 (positive)'])
axes[1].legend()

# Categorical (softmax output)
classes = ['airplane', 'car', 'bird', 'cat', 'deer']
logits = np.array([0.5, 2.1, 0.3, 1.8, 0.2])
probs = np.exp(logits) / np.exp(logits).sum()
bars = axes[2].bar(classes, probs, color='steelblue', alpha=0.8)
bars[1].set_color('coral')
axes[2].set_title('Categorical (softmax output)')
axes[2].set_ylabel('Probability')
for bar, p in zip(bars, probs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{p:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Softmax probabilities sum: {probs.sum():.6f} — must be exactly 1")
print(f"Most likely class: {classes[probs.argmax()]} ({probs.max():.2%})")

---
## 3. Bayes' Theorem

### Intuition

You have a belief. You see new evidence. How do you update your belief?

**Example:** You think there's a 1% chance you have a rare disease. You take a test that's 99% accurate. It comes back positive. Now what?

Intuition says: 99% accurate test + positive result → probably have the disease.

Bayes says: **not so fast.** The disease is rare — only 1 in 100 people have it. Most positive tests are actually false positives from the 99 healthy people. You probably don't have it.

This is the power of Bayes — it forces you to account for how common or rare something is before jumping to conclusions.

### Formula

$$\boxed{P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}}$$

| Term | Name | Meaning |
|---|---|---|
| $P(A)$ | **Prior** | Your belief before seeing evidence |
| $P(B \mid A)$ | **Likelihood** | How probable is the evidence if A is true? |
| $P(A \mid B)$ | **Posterior** | Your updated belief after seeing evidence |
| $P(B)$ | **Evidence** | How probable is the evidence overall? |

The key insight: **posterior ∝ likelihood × prior**

### In ML

Training a neural network is Bayesian updating at scale. You start with random weights (prior). You see training data (evidence). Gradient descent updates weights to better explain the data (posterior). Every step is a Bayes update.

In [ ]:
P_disease = 0.01
P_no_disease = 1 - P_disease
P_pos_given_disease = 0.99
P_pos_given_no_disease = 0.05

# Law of total probability
P_positive = (P_pos_given_disease * P_disease +
              P_pos_given_no_disease * P_no_disease)

# Bayes' theorem
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_positive

print("BAYES' THEOREM — Medical Test")
print(f"Prior P(disease):               {P_disease:.2%}")
print(f"Test sensitivity P(pos|disease): {P_pos_given_disease:.2%}")
print(f"False positive P(pos|healthy):   {P_pos_given_no_disease:.2%}")
print(f"\nPosterior P(disease | positive): {P_disease_given_pos:.2%}")
print(f"\nDespite 99% accurate test, only {P_disease_given_pos:.1%} chance of disease!")
print(f"The rare prior (1%) dominates.")

# Imagine 10,000 people
n = 10000
true_pos  = int(n * P_disease * P_pos_given_disease)
false_pos = int(n * P_no_disease * P_pos_given_no_disease)
print(f"\nOut of {n:,} people tested:")
print(f"  True positives (sick):    {true_pos}")
print(f"  False positives (healthy): {false_pos}")
print(f"  → Of all positives, only {true_pos/(true_pos+false_pos):.1%} are actually sick")

---
## 4. Expectation and Variance

### Intuition

Roll a die many times. On average, what number do you get? That's the **expectation** — the long-run average.

But two distributions can have the same average and look very different. **Variance** measures the spread.

Think of two archers:
- Archer A: always hits near the center — low variance, reliable
- Archer B: hits everywhere on the target — high variance, unpredictable

Both might average to the center, but Archer A is better.

### Expectation

Weighted average of all values, weighted by their probability:
$$\mathbb{E}[X] = \sum_x x \cdot P(X=x) \quad \text{(discrete)}$$
$$\mathbb{E}[X] = \int x \cdot p(x)\,dx \quad \text{(continuous)}$$

**Key property (linearity):** $\mathbb{E}[aX + b] = a\mathbb{E}[X] + b$

### Variance

Average squared distance from the mean:
$$\text{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

**Standard deviation** $\sigma = \sqrt{\text{Var}(X)}$ — same units as X, easier to interpret.

### In ML

- **Loss function** = $\mathbb{E}[\text{error}]$ — expected error over all training examples
- **Bias**: model expectation far from truth → underfitting
- **Variance**: model output varies wildly with different data → overfitting

In [ ]:
outcomes = np.array([1, 2, 3, 4, 5, 6])
probs = np.ones(6) / 6

expectation = np.sum(outcomes * probs)
variance = np.sum((outcomes - expectation)**2 * probs)
std = np.sqrt(variance)

print("DIE ROLL")
print(f"E[X] = {expectation:.4f}  (true value: 3.5)")
print(f"Var(X) = {variance:.4f}  (true value: {35/12:.4f})")
print(f"Std(X) = {std:.4f}")

print("\nLaw of Large Numbers — more samples → closer to true expectation:")
for n in [10, 100, 1000, 100000]:
    samples = np.random.choice(outcomes, size=n, p=probs)
    print(f"  n={n:>7,}: sample mean = {samples.mean():.4f}")
print(f"  True E[X] =       {expectation:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.linspace(-10, 10, 300)
for sigma, label in [(0.5, 'Low variance σ=0.5'), (2, 'Medium σ=2'), (4, 'High variance σ=4')]:
    ax.plot(x, stats.norm.pdf(x, 0, sigma), label=label, linewidth=2)
ax.axvline(0, color='black', linestyle='--', alpha=0.5, label='Same mean=0 for all')
ax.set_title('Same Mean, Different Variance')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Joint, Marginal, and Conditional Probability

### Intuition

So far: one random variable at a time. ML involves two things at once: an image and its label. A word and its context.

Three ways to describe probability with two variables:

**Scenario:** Roll two dice — $X$ = first die, $Y$ = second die.

- **Joint** $P(X=3, Y=5)$: both dice show those exact values → 1/36
- **Marginal** $P(X=3)$: first die shows 3, regardless of second → 1/6
- **Conditional** $P(X=3 \mid Y=5)$: first die shows 3, given second shows 5 → still 1/6 (dice are independent)

### Formulas

$$P(X=x, Y=y) \quad \text{Joint — both happen}$$

$$P(X=x) = \sum_y P(X=x, Y=y) \quad \text{Marginal — sum out Y}$$

$$P(X=x \mid Y=y) = \frac{P(X=x, Y=y)}{P(Y=y)} \quad \text{Conditional — given Y}$$

**Independence** — knowing Y tells you nothing about X:
$$X \perp Y \iff P(X, Y) = P(X) \cdot P(Y)$$

### In ML

| Model type | What it learns | Example |
|---|---|---|
| **Generative** | Joint $P(\text{image}, \text{label})$ | VAE, GAN, Diffusion |
| **Discriminative** | Conditional $P(\text{label} \mid \text{image})$ | ResNet, ViT, BERT |

In [ ]:
joint = np.ones((6, 6)) / 36
marginal_X = joint.sum(axis=1)
marginal_Y = joint.sum(axis=0)

print("JOINT: P(X=1, Y=1) =", f"{joint[0,0]:.4f}  (should be 1/36 = {1/36:.4f})")
print("MARGINAL: P(X=1) =", f"{marginal_X[0]:.4f}  (should be 1/6 = {1/6:.4f})")

y_idx = 3  # Y=4
conditional = joint[:, y_idx] / marginal_Y[y_idx]
print(f"\nCONDITIONAL P(X | Y=4):")
for i, p in enumerate(conditional):
    print(f"  P(X={i+1} | Y=4) = {p:.4f}")
print(f"  Sum = {conditional.sum():.4f} — must be 1")
print(f"  Same as marginal? {np.allclose(conditional, marginal_X)} — dice are independent")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im = axes[0].imshow(joint, cmap='Blues')
axes[0].set_title('Joint P(X,Y)')
axes[0].set_xlabel('Y'); axes[0].set_ylabel('X')
plt.colorbar(im, ax=axes[0])

axes[1].bar(range(1,7), marginal_X, color='steelblue', alpha=0.8)
axes[1].set_title('Marginal P(X)')
axes[1].set_ylim(0, 0.25)

axes[2].bar(range(1,7), conditional, color='coral', alpha=0.8)
axes[2].set_title('Conditional P(X|Y=4)')
axes[2].set_ylim(0, 0.25)

plt.tight_layout()
plt.show()

---
## 6. Maximum Likelihood Estimation (MLE)

### Intuition

You observe some data. You believe the data came from a distribution with unknown parameters $\theta$.

**MLE asks: what parameters $\theta$ make the observed data most probable?**

**Analogy:** You measure 10 people's heights: 168, 172, 165, 170, 174... You assume heights follow a Gaussian. MLE finds the $\mu$ and $\sigma$ that best explain those measurements.

For a Gaussian, MLE gives: $\mu_{MLE}$ = sample mean, $\sigma_{MLE}$ = sample std. Intuitive — but MLE derives this rigorously from first principles.

### Formula

**Likelihood** — probability of seeing all your data under parameters $\theta$:
$$\mathcal{L}(\theta) = \prod_{i=1}^n p(x^{(i)};\theta)$$

**Problem:** Multiplying many small probabilities causes numerical underflow.

**Solution:** Take the log (log-likelihood). The maximum is in the same place:
$$\hat{\theta}_{MLE} = \arg\max_\theta \sum_{i=1}^n \log p(x^{(i)};\theta)$$

### In ML — Training IS MLE

- Model computes $P(y^{(i)} \mid x^{(i)}; \theta)$ — likelihood of correct label
- Maximizing log-likelihood = minimizing negative log-likelihood
- **CrossEntropyLoss = negative log-likelihood**
- Every `loss.backward()` call is doing MLE

In [ ]:
np.random.seed(42)
true_mu, true_sigma = 5.0, 2.0
data = np.random.normal(true_mu, true_sigma, 500)

mu_mle = data.mean()
sigma_mle = data.std()

print("MLE FOR GAUSSIAN")
print(f"True:  μ = {true_mu:.2f}, σ = {true_sigma:.2f}")
print(f"MLE:   μ = {mu_mle:.4f}, σ = {sigma_mle:.4f}")

mu_range = np.linspace(2, 8, 100)
log_likelihoods = [np.sum(stats.norm.logpdf(data, mu, sigma_mle)) for mu in mu_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mu_range, log_likelihoods, 'b-', linewidth=2)
axes[0].axvline(mu_mle, color='red', linestyle='--', label=f'MLE μ={mu_mle:.2f}')
axes[0].axvline(true_mu, color='green', linestyle=':', label=f'True μ={true_mu:.2f}')
axes[0].set_xlabel('μ')
axes[0].set_ylabel('Log-likelihood')
axes[0].set_title('Log-likelihood peaks at MLE')
axes[0].legend()

axes[1].hist(data, bins=40, density=True, alpha=0.6, label='Data')
x = np.linspace(data.min(), data.max(), 300)
axes[1].plot(x, stats.norm.pdf(x, mu_mle, sigma_mle), 'r-', linewidth=2, label='MLE fit')
axes[1].plot(x, stats.norm.pdf(x, true_mu, true_sigma), 'g--', linewidth=2, label='True')
axes[1].set_title('MLE fit vs true distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nML CONNECTION:")
p_correct = 0.70
cross_entropy = -np.log(p_correct)
print(f"Model predicts P(correct class) = {p_correct}")
print(f"CrossEntropy = -log({p_correct}) = {cross_entropy:.4f}")
print(f"Minimizing CrossEntropy = Maximizing likelihood of correct label = MLE")

---
## 7. KL Divergence

### Intuition

You have two distributions. How different are they?

**KL Divergence** measures how much information is lost when you use distribution $Q$ to approximate distribution $P$.

**Analogy:** $P$ is the real weather (complex, accurate). $Q$ is your simplified model of weather. KL divergence measures how surprised you'd be, on average, if you predicted using $Q$ but reality followed $P$.

If $Q = P$: no surprise, KL = 0.  
If $Q$ is very different from $P$: lots of surprise, KL is large.

### Formula

$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

**Three key properties:**
1. Always non-negative: $D_{KL} \geq 0$
2. Zero only when identical: $D_{KL} = 0 \iff P = Q$
3. **Not symmetric:** $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$ — order matters!

### In ML — Everywhere

| Where | How |
|---|---|
| **Cross-entropy loss** | Minimizing CE = minimizing KL from true distribution |
| **VAE** | Loss = reconstruction + $D_{KL}(q(z|x) \| p(z))$ |
| **RLHF** | KL penalty keeps fine-tuned LLM close to base model |
| **Distillation** | Student minimizes KL from teacher's output distribution |

In [ ]:
def kl_divergence(P, Q, eps=1e-10):
    P = np.array(P, dtype=float) + eps
    Q = np.array(Q, dtype=float) + eps
    P /= P.sum(); Q /= Q.sum()
    return np.sum(P * np.log(P / Q))

P_true  = np.array([0, 0, 1, 0, 0])                 # true label: class 2
Q_great = np.array([0.01, 0.01, 0.96, 0.01, 0.01])  # confident + correct
Q_ok    = np.array([0.1, 0.1, 0.6, 0.1, 0.1])       # uncertain + correct
Q_bad   = np.array([0.2, 0.2, 0.2, 0.2, 0.2])       # completely flat
Q_wrong = np.array([0.01, 0.01, 0.01, 0.01, 0.96])  # confident + wrong

print("KL DIVERGENCE — True label = class 2")
print(f"KL(P || Q_great) = {kl_divergence(P_true, Q_great):.4f}  confident + correct")
print(f"KL(P || Q_ok)    = {kl_divergence(P_true, Q_ok):.4f}  uncertain + correct")
print(f"KL(P || Q_bad)   = {kl_divergence(P_true, Q_bad):.4f}  completely flat")
print(f"KL(P || Q_wrong) = {kl_divergence(P_true, Q_wrong):.4f}  confident + WRONG (worst)")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
preds  = [Q_great, Q_ok, Q_bad, Q_wrong]
titles = ['Great', 'OK', 'Flat', 'Wrong']
kls    = [kl_divergence(P_true, q) for q in preds]
classes = ['C0','C1','C2','C3','C4']

for ax, q, title, kl in zip(axes, preds, titles, kls):
    bars = ax.bar(classes, q, alpha=0.8, color='steelblue')
    bars[2].set_color('green')
    ax.set_title(f'{title}\nKL={kl:.3f}')
    ax.set_ylim(0, 1.05)

plt.suptitle('True label = C2 (green). Lower KL = better prediction.', y=1.02)
plt.tight_layout()
plt.show()

P = np.array([0.9, 0.1])
Q = np.array([0.5, 0.5])
print(f"\nAsymmetry: KL(P||Q)={kl_divergence(P,Q):.4f},  KL(Q||P)={kl_divergence(Q,P):.4f}")

---
## 8. Central Limit Theorem (CLT)

### Intuition

Roll one die: result is uniform — flat, not bell-shaped.

Roll 10 dice and take the average: starts clustering around 3.5.

Roll 100 dice and take the average: tightly clustered, looks like a bell curve.

**The Central Limit Theorem:** No matter what distribution you start with, the average of many independent samples from it approaches a Gaussian.

This is one of the most remarkable facts in mathematics. It's why Gaussians appear everywhere in nature — because most things in the world are sums of many small independent effects.

### Formula

Let $X_1, ..., X_n$ be independent samples from any distribution with mean $\mu$ and variance $\sigma^2$.

As $n \to \infty$:
$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{\text{distribution}} \mathcal{N}\left(\mu, \frac{\sigma^2}{n}\right)$$

Key insight: variance of the mean = $\sigma^2/n$. More samples → tighter estimate.

### In ML

| Where | Why CLT matters |
|---|---|
| **Mini-batch gradient** | Average of 32 gradients ≈ Gaussian — justifies SGD theory |
| **Gaussian noise** | Many real errors are sums of small effects → Gaussian |
| **Batch normalization** | Assumes layer outputs roughly Gaussian — CLT is why this works |

In [ ]:
np.random.seed(42)
n_experiments = 10000
sample_sizes = [1, 2, 10, 50]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, n in enumerate(sample_sizes):
    # Exponential distribution — very non-Gaussian, highly skewed
    raw = np.random.exponential(scale=1.0, size=(n_experiments, n))
    means = raw.mean(axis=1)

    axes[0, col].hist(raw[:, 0], bins=40, density=True, color='coral', alpha=0.8)
    axes[0, col].set_title(f'n={n}: one sample\n(exponential — skewed)')

    axes[1, col].hist(means, bins=50, density=True, color='steelblue', alpha=0.8)
    x = np.linspace(means.min(), means.max(), 200)
    axes[1, col].plot(x, stats.norm.pdf(x, 1.0, 1.0/np.sqrt(n)), 'r-', linewidth=2, label='CLT prediction')
    axes[1, col].set_title(f'n={n}: sample means\nskewness={stats.skew(means):.3f}')
    if col == 0:
        axes[1, col].legend()

axes[0, 0].set_ylabel('Density')
axes[1, 0].set_ylabel('Density')
plt.suptitle('CLT: Exponential → Gaussian as n grows', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("As n grows: skewness → 0 (becoming Gaussian), spread → narrower (variance = σ²/n)")
print("\nMini-batch intuition:")
print("True gradient = average gradient over all training examples")
print("Mini-batch gradient = average over 32 examples ≈ Gaussian noise around true gradient")
print("This noise helps escape sharp local minima — a feature, not a bug.")

---
## Summary

| # | Concept | Core Idea | ML Connection |
|---|---|---|---|
| 1 | Random Variable | Unknown number from a random process | Model output = distribution over classes |
| 2 | Distributions | Shape of probability across values | Choice of distribution = choice of loss function |
| 3 | Bayes' Theorem | Evidence updates prior belief | Training = posterior inference |
| 4 | Expectation + Variance | Average and spread | Loss = expected error; bias-variance tradeoff |
| 5 | Joint/Marginal/Conditional | Probability with two variables | Generative vs discriminative models |
| 6 | MLE | Parameters that best explain data | CrossEntropyLoss = negative log-likelihood |
| 7 | KL Divergence | Distance between distributions | VAE, RLHF, distillation |
| 8 | CLT | Averages → Gaussian | Mini-batch gradients, BatchNorm |

---

### The Full Story of Training a Neural Network

1. Data comes from a **distribution** (concept 2)
2. We model label given input as a **conditional probability** (concept 5)
3. We choose parameters via **MLE** — maximize log P(correct label | image) (concept 6)
4. Minimizing CrossEntropy = minimizing **KL divergence** from true labels (concept 7)
5. Each gradient step is a **Bayes update** — evidence moves our beliefs (concept 3)
6. Mini-batch gradients are approximately Gaussian by the **CLT** (concept 8)
7. The loss measures **expected error** (concept 4) over the **random variable** output (concept 1)